In [1]:
import scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# import equivariant steerable cnn
import escnn.gspaces
import escnn.nn

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import from_numpy

from tqdm import tqdm
# Checking if you have a GPU and CUDA set up -- if not, training will take longer by running on the CPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# device = 'cpu'
# Set the plotting style
#plt.style.use('seaborn-poster')

from sbi.inference import SNPE_C
from sbi import utils
from sbi.utils.get_nn_models import posterior_nn

from os import listdir
from os.path import isfile, join

from copy import deepcopy

from getdist import MCSamples, plots

In [2]:
ens = 20 # Turn this up
onlyfiles = [f[:-4] for f in listdir('.') if f[-3:] == 'npy']
onlyfiles_split = np.array([s.split('_') for s in onlyfiles])
stds = onlyfiles_split[:,8].astype(np.float32)
corrlens = onlyfiles_split[:,10].astype(np.float32)

imgs = np.zeros((ens*len(onlyfiles),100,100))
for i, name in enumerate(onlyfiles):
    img_stack = np.load(name+'.npy')
    imgs[i*ens:(i+1)*ens] = img_stack


imgs = from_numpy(np.reshape(imgs.astype(np.float32),((ens*len(onlyfiles),1,100,100)))).to('cuda')

In [3]:
stds_tiled = (np.ones((ens,len(stds)),dtype=np.float32)*stds).T.flatten()
corrlens_tiled = (np.ones((ens,len(corrlens)),dtype=np.float32)*corrlens).T.flatten()

# building one theta vector
theta = from_numpy(np.vstack((stds_tiled,corrlens_tiled)).T).to('cuda')

In [9]:
# set up C3NN
bandlimit = 6
gspace = escnn.gspaces.rot2dOnR2(N=-1, maximum_frequency=bandlimit)


""" Implements the non-linear convolution operation using the recursive formula."""
class NonlinearConv(nn.Module):
    def __init__(self, in_channels, num_filts, filter_size, order):
        super().__init__()
        self.order = order
        self.in_channels = in_channels
        self.num_filts = num_filts
        self.filter_size = filter_size

        """
        # Initialize filter similar as to how Pytorch initializes nn.Conv2d, but all positive
        # This is the old way of initializing C3NN filters without rotational equivariance/invariance
        weight_sample_k = 1.0 / np.sqrt(filter_size**2 * in_channels)
        init_filt = (2 * weight_sample_k * torch.rand(num_filts, in_channels, filter_size, filter_size, dtype=torch.float32))
        self.conv_filt = torch.nn.Parameter(init_filt)
        # Initialize some slices of the filter to be 0
        with torch.no_grad():
            self.conv_filt[:, :, int(self.filter_size/2)-2:int(self.filter_size/2)+3, int(self.filter_size/2)-2:int(self.filter_size/2)+3]=0.0
        """
        
        # Initialize the escnn filter
        feat_type_in = escnn.nn.FieldType(gspace, self.in_channels*[gspace.trivial_repr])
        feat_type_out = escnn.nn.FieldType(gspace, self.num_filts*[gspace.trivial_repr])
        # THIS LAYER KILLS DEEPCOPY
        self.conv = escnn.nn.R2Conv(in_type=feat_type_in, out_type=feat_type_out, bias=False, kernel_size=self.filter_size) 
        
        # to enable deepcopy
        self.detached = False
        
    def forward(self, x):
        # We will work with strictly positive filters for easier interpretation!
        # foll
        if self.detached == False:
            self.abs_filt = self.conv.expand_parameters()[0].abs()
        elif self.detached == True:
            pass
        
        # Convolutions of powers of the input with powers of the absolute valued filter
        conv_powers = [F.conv2d(x ** k, self.abs_filt ** k, stride=1, padding=int((self.filter_size-1)/2.0)) for k in range(1, self.order + 1)]
        
        #troubleshooting:
        #conv_powers = [self.convnormal(x ** k) for k in range(1, self.order + 1)]
        
        # C_0 is defined to be one everywhere
        corrs = [torch.ones_like(conv_powers[0])]

        # Compute each order of C_n recursively
        for n in range(1, self.order + 1):
            Cn = torch.zeros_like(conv_powers[0])
            for k in range(n):
                Cn += (-1) ** k * conv_powers[k] * corrs[n - k - 1]
            Cn /= n
            corrs.append(Cn)

        # Stack correlator maps along the channel dimension, dropping the zeroth order (C_0)
        # Here the channel dimension refers to the number of channels
        #return torch.cat(corrs[1:], dim=-3)
        return torch.cat(corrs[1:], dim=-3)

    def _detach(self):
        del self.conv
        self.abs_filt = self.abs_filt.detach()
        self.detached = True


""" Does a simple average over the last two dimensions of the input."""
class Mean2d(nn.Module):
    def __init__(self):
        super().__init__()
        
    def forward(self, x):
        return x.mean((-2, -1))


""" The full CCNN is Input -> NonlinearConv -> Spatial Avg -> BatchNorm -> Logistic. The way it is constructed
    here assumes a binary classification task, but this can be extended to the multiclass case."""
class CCNN(nn.Module):
    def __init__(self, in_channels, num_filts, filter_size, order, absbeta=False, slice=False, pool=False):
        super().__init__()
        self.in_channels = in_channels
        self.num_filts = num_filts
        self.filter_size = filter_size
        self.order = order
        self.absbeta = absbeta

        # the model architecture with escnn
        self.corr_conv = NonlinearConv(self.in_channels, self.num_filts, self.filter_size, self.order)
        self.spatial_mean = Mean2d()
        # I preserve the batch normalization layer here so that we can scale the compressed summary statistics to the same order of magnitude
        # This may help the numerical stability of inference in the 2nd part
        self.bn = nn.BatchNorm1d(num_filts * order, affine=True, eps=3.185289e-35, dtype=torch.float32)


        #troubleshooting
        #self.conv = Conv2d(1,1,11)
    
    def forward(self, x, chop=True):    
        
        x = self.corr_conv(x)
        #x = self.conv(x)
        x = self.spatial_mean(x)
        if chop == False:
            x = self.bn(x)


        return x

    def detach(self):
        self.corr_conv._detach()


#=== 2) Set up some things for training ===#
# Number of times the model will see the full dataset
epochs = 5
# The initial learning rate -- an important hyperparameter
lr = 0.505631 
#lr = 0.5
# The strength of the regularization loss -- we can make this very strong in these easy toy examples
gamma = 0.030383
# The number of input channels 
in_channels = 1
# The number of filters to learn in our model
num_filts = 1
# The spatial length of the filters
filter_size = 11
# The highest order of correlation measured
order = 2
# Make our model -- for simplicity, let's train a two-filter model

In [10]:
ccnn = CCNN(in_channels=in_channels, num_filts=num_filts, filter_size=filter_size, order=order, absbeta=False).to(device='cuda')

# Set up prior for sbi
#lower = np.array([1.,51.])
#upper = np.array([5.,32.])
lower = np.array([.8,3.])
upper = np.array([4.,30.])
prior = utils.BoxUniform(low=from_numpy(lower), high=from_numpy(upper),device='cuda')


#set up sbi architecture
density_estimator_custom = posterior_nn(model="maf", hidden_features=32, num_transforms=3, device='cuda', embedding_net=ccnn)
inference = SNPE_C(prior, device='cuda', density_estimator=density_estimator_custom)

# joint training
density_estimator = inference.append_simulations(theta, imgs).train(training_batch_size=20,max_num_epochs=5)
# DANGER DANGER
ccnn.detach()
# now deepcopy works:
test = deepcopy(density_estimator._embedding_net)

 Training neural network. Epochs trained: 6DANGER!!!! No more deepycopy after training!!!


In [11]:
# build posterior and sample
posterior = inference.build_posterior(deepcopy(density_estimator))
posterior_samples = posterior.sample((100000,), x=imgs[10]).cpu()

Drawing 100000 posterior samples:   0%|          | 0/100000 [00:00<?, ?it/s]

In [12]:
density_estimator._embedding_net(imgs[:30])

tensor([[ 2.4360, 17.2711],
        [ 0.1118, 16.5893],
        [ 1.7984, 15.3319],
        [ 0.0318, 13.1808],
        [ 0.4888, 10.5450],
        [ 0.9914, 18.2827],
        [ 1.0751, 22.4302],
        [-0.2420, 10.1953],
        [ 0.5171, 12.3140],
        [-0.9727, 19.4341],
        [ 0.8083, 25.4277],
        [-0.8412, 15.3725],
        [-1.8912, 14.8996],
        [-1.1839, 28.2330],
        [-2.3367, 16.0295],
        [ 1.5953, 14.8014],
        [ 0.1530, 13.1272],
        [ 1.3353, 12.8751],
        [-0.8349, 15.5302],
        [ 0.0438,  5.0023],
        [-0.2768, 14.7528],
        [ 0.7543, 16.2187],
        [-1.0784, 12.6556],
        [-1.1090, 16.9932],
        [ 0.2567, 15.3343],
        [-0.3685, 12.2802],
        [-0.9386, 17.3170],
        [ 0.4431, 16.1562],
        [ 1.3418, 13.9125],
        [-0.0820, 15.6493]], device='cuda:0')

In [13]:
names = np.array(['\sigma','corrlen'])
params = [1.,15.]


samples = MCSamples(samples=posterior_samples.numpy(), names=names, labels=names, settings={'ignore_rows':.2},label='sbi, Likelihood-Free')
g = plots.get_subplot_plotter()
#g.settings.lab_fontsize = 17
#g.settings.axes_fontsize = 14.3
#g.settings.legend_fontsize = 14.5
g.triangle_plot([samples], names, title_limit=1, filled=True,
                     markers={'\sigma':params[0],'corrlen':params[1]},
                     marker_args={'c':'black'})

plt.show()
plt.savefig('test.png')

Removed 0.2 as burn in


In [63]:
print(stds)

[1.        2.010101  2.010101  1.        1.        1.5050505 1.5050505
 1.        2.010101  1.        1.5050505 1.5050505 1.        1.5050505
 1.        1.        2.010101  1.        1.5050505 1.5050505 2.010101
 1.5050505 1.        1.5050505 1.5050505]
